# N-gram Language Modeling on Persian TinyStories

This notebook implements N-gram language models on the **TinyStories-Farsi** dataset. It covers data cleaning, BPE tokenization, model training for 2-gram, 4-gram and 8-gram models, text generation, perplexity evaluation, smoothing techniques (Laplace, Interpolation, Backoff) and temperature-based sampling.

In [1]:
import os
import re
import math
import random
from dotenv import load_dotenv
from tokenizers import Tokenizer
from datasets import load_dataset
from tokenizers.models import BPE
from huggingface_hub import login
from collections import defaultdict
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace


random.seed(42)
load_dotenv()
login(os.getenv("HUGGINGFACE_TOKEN"))

## 1. Data Cleaning & Tokenization

The **TinyStories-Farsi** dataset is loaded. The Persian training split undergoes normalization:

- Replace Arabic characters `ي` and `ك` with their Persian equivalents `ی` and `ک`.
- Convert English digits `0-9` to Persian digits `۰-۹`.
- Replace the zero-width non-joiner (ZWNJ, نیم‌فاصله) with a regular space.
- Collapse multiple whitespace characters into a single space and strip edges.

A **Byte-Pair Encoding (BPE)** tokenizer with a vocabulary size of 5,000 is then trained on the cleaned corpus.

In [2]:
dataset = load_dataset("taesiri/TinyStories-Farsi")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = text.translate(str.maketrans("0123456789", "۰۱۲۳۴۵۶۷۸۹"))
    text = text.replace("\u200c", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def analyze_cleaning_changes(text):
    if not isinstance(text, str):
        text = ""
    return {
        "arabic_to_persian_y": len(re.findall("ي", text)),
        "arabic_to_persian_k": len(re.findall("ك", text)),
        "english_digits_to_persian": len(re.findall(r"[0-9]", text)),
        "half_to_space": text.count("\u200c"),
        "initial_length": len(text),
        "final_length": len(clean_text(text)),
    }


all_stats = defaultdict(int)
for i in range(len(train_dataset)):
    stat = analyze_cleaning_changes(train_dataset[i].get("Persian", ""))
    for key, value in stat.items():
        all_stats[key] += value

total_initial = all_stats["initial_length"]
total_final = all_stats["final_length"]
removed = max(0, total_initial - total_final)

print("===== Preprocessing Summary (train set) =====")
print(f"Total initial length (chars): {total_initial:,}")
print(f"Total final length (chars):   {total_final:,}")
print(f"Total removed chars:          {removed:,} ({100 * removed / total_initial:.2f}%)")
print("\n-- Detailed counts --")
print(f"Arabic y  -> Persian ی : {all_stats['arabic_to_persian_y']:,}")
print(f"Arabic k  -> Persian ک : {all_stats['arabic_to_persian_k']:,}")
print(f"English digits -> Persian : {all_stats['english_digits_to_persian']:,}")
print(f"Half space -> space       : {all_stats['half_to_space']:,}")

# Apply cleaning
cleaned_train = train_dataset.map(lambda x: {"text": clean_text(x["Persian"])})
cleaned_validation = validation_dataset.map(lambda x: {"text": clean_text(x["Persian"])})

README.md:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

InProgress-TinyStoriesV2-GPT4-train-Tran(…): reconstructing file:   0%|          |  0.00B /  271MB            

InProgress-TinyStoriesV2-GPT4-train-Tran(…): downloading bytes:           |  0.00B            

TinyStoriesV2-GPT4-valid-Translated-To-F(…): reconstructing file:   0%|          |  0.00B / 59.9MB            

TinyStoriesV2-GPT4-valid-Translated-To-F(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/124411 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27630 [00:00<?, ? examples/s]

===== Preprocessing Summary (train set) =====
Total initial length (chars): 95,545,837
Total final length (chars):   94,859,981
Total removed chars:          685,856 (0.72%)

-- Detailed counts --
Arabic y  -> Persian ی : 51,097
Arabic k  -> Persian ک : 15,147
English digits -> Persian : 424
Half space -> space       : 1,116,558


Map:   0%|          | 0/124411 [00:00<?, ? examples/s]

Map:   0%|          | 0/27630 [00:00<?, ? examples/s]

In [3]:
SPECIAL_TOKENS = ["[UNK]", "[PAD]", "[SOS]", "[EOS]"]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=SPECIAL_TOKENS, vocab_size=5000)
tokenizer.train_from_iterator(cleaned_train["text"], trainer)

tokenized_dataset = cleaned_train.map(lambda x: {"tokens": tokenizer.encode(x["text"]).tokens})
tokenized_validation = cleaned_validation.map(
    lambda x: {"tokens": tokenizer.encode(x["text"]).tokens}
)

print(f"Number of cleaned train samples: {len(cleaned_train)}")
print("\nSample sentence:\n", cleaned_train[10]["text"][:200], "...")
print("\nTokens:")
sample_tokens = tokenizer.encode(cleaned_train[10]["text"]).tokens
for idx, tok in enumerate(sample_tokens[:60]):
    print(f"{tok}, ", end="")
    if idx % 20 == 19:
        print()

Map:   0%|          | 0/124411 [00:00<?, ? examples/s]

Map:   0%|          | 0/27630 [00:00<?, ? examples/s]

Number of cleaned train samples: 124411

Sample sentence:
 تام و لیلی دوقلوهایی بودند که دوست داشتند در پارک بازی کنند. یک روز آنها یک گیره بزرگ ساخته شده از بوته های سبز دیدند. آنها می خواستند آن را امتحان کنند و راه خروج را پیدا کنند. آنها درون گیره دویدند  ...

Tokens:
تام, و, لیلی, دوقلوهایی, بودند, که, دوست, داشتند, در, پارک, بازی, کنند, ., یک, روز, آنها, یک, گیره, بزرگ, ساخته, 
شده, از, بوته, های, سبز, دیدند, ., آنها, می, خواستند, آن, را, امتحان, کنند, و, راه, خروج, را, پیدا, کنند, 
., آنها, درون, گیره, دویدند, و, مسیر, ها, را, دنبال, کردند, ., گاهی, به, چپ, می, پیچ, یدند, ،, گاهی, 


## 2. N-gram Language Models

An `NGramModel` class builds count-based N-gram models. Start-of-sentence padding (`[SOS]`) and end-of-sentence (`[EOS]`) markers are added to each sequence. Three models with **n = 2, 4, 8** are trained on the tokenized Persian corpus, and each model generates a 100-token sample.

In [4]:
class NGramModel:
    """Count-based N-gram language model with proper context tracking."""

    def __init__(self, n):
        self.n = n
        self.ngram_counts = defaultdict(lambda: defaultdict(int))
        self.context_counts = defaultdict(int)
        self.START_SYMBOL = "[SOS]"
        self.END_SYMBOL = "[EOS]"

    # ---------- training ----------
    def _iter_ngrams(self, token_sequence):
        padding = [self.START_SYMBOL] * (self.n - 1)
        full = padding + list(token_sequence) + [self.END_SYMBOL]
        for i in range(len(full) - self.n + 1):
            yield tuple(full[i : i + self.n])

    def train(self, tokenized_data):
        for item in tokenized_data:
            for ngram in self._iter_ngrams(item["tokens"]):
                context, target = ngram[:-1], ngram[-1]
                self.ngram_counts[context][target] += 1
                self.context_counts[context] += 1

    # ---------- sampling ----------
    def generate(self, num_tokens=100):
        context = tuple([self.START_SYMBOL] * (self.n - 1))
        out = []
        guard = 0
        while len(out) < num_tokens and guard < num_tokens * 20:
            guard += 1
            dist = self.ngram_counts.get(context)
            if not dist:
                context = tuple([self.START_SYMBOL] * (self.n - 1))
                continue
            options = list(dist.keys())
            weights = list(dist.values())
            token = random.choices(options, weights=weights, k=1)[0]
            if token == self.END_SYMBOL:
                context = tuple([self.START_SYMBOL] * (self.n - 1))
            else:
                out.append(token)
                context = tuple(list(context[1:]) + [token])
        return out


for n in (2, 4, 8):
    print(f"\n=== Training {n}-gram model ===")
    model = NGramModel(n)
    model.train(tokenized_dataset)
    tokens = model.generate(100)
    print(f"Sample sentence created by {n}-gram:")
    print(tokenizer.decode(tokenizer.encode(" ".join(tokens)).ids), "\n\n")


=== Training 2-gram model ===
Sample sentence created by 2-gram:
باری روزی ، من احمق ، بله ، به اسم تیم تمام روز ، مکس به پارک گذرانده بودند . مادرشان در حیاط خانه خورد . او یک غافلگیری را ملاقات کردند . مرغ و صدای در بازار دیدند . ماشین را با دوستانش گفت : « بیا که یک سگ های زیبا آمدند . عروسک ها نزدیک تر است شجاع احساس می افتد ." دختر کوچک ، می خواست مثل جرقه شروع به هم دوست باشیم و مادرش گفت : " نه خیلی ناراحت شد محبوب ترین ش گذاشت . لیلی جعبه رو به تام ‫ پس دسر هایی 



=== Training 4-gram model ===
Sample sentence created by 4-gram:
یک روزی یک سگ کوچولو به نام لیلی زندگی می کرد . اسپات دوست داشت کل روز بخوابد . یک روز ، دختربچه ای به نام تیم روی سنگ بزرگی دید و دوست داشت با تلفن بازی حرف زدند . تیم گفت : « من از فرو ت نی به خانه رسیدند ، مامان گفت : " تیم ، تو باید آرام بگیری ." اردک موافقت کرد : " می دو نی این کلید چی رو باز می کنه پیدا کنی و روی دیوار گذاشت . سو گریه کرد و گریه کرد . هر روز مادرش اون رو بردن به 



=== Training 8-gram model ===
Sample sentence created by 8-gram

## 3. Perplexity Evaluation

Perplexity is computed on the **validation set** for each N-gram order, both with and without **Laplace (add-one) smoothing**. Without smoothing, any unseen N-gram yields `inf`; with Laplace smoothing the perplexity becomes finite.

In [5]:
class NGramPerplexity(NGramModel):
    """NGramModel extended with perplexity computation."""

    def perplexity(self, dataset, vocab_size=None, smoothing="none"):
        total_log_prob = 0.0
        total_tokens = 0

        for item in dataset:
            for ngram in self._iter_ngrams(item["tokens"]):
                context, target = ngram[:-1], ngram[-1]
                total_tokens += 1
                c_ctx = self.context_counts.get(context, 0)
                c_ng = self.ngram_counts[context].get(target, 0) if c_ctx else 0

                if smoothing == "laplace":
                    prob = (c_ng + 1) / (c_ctx + vocab_size)
                else:  # no smoothing
                    if c_ctx == 0 or c_ng == 0:
                        return float("inf")
                    prob = c_ng / c_ctx

                total_log_prob += math.log(prob)

        return math.exp(-total_log_prob / total_tokens)


V = tokenizer.get_vocab_size()
perplexity_models = {}
for n in (2, 4, 8):
    model = NGramPerplexity(n)
    model.train(tokenized_dataset)
    perplexity_models[n] = model

print("Calculating perplexity on validation data...\n")
print(f"{'n':>3} | {'unsmoothed':>15} | {'Laplace (add-1)':>18}")
print("-" * 44)
for n, model in perplexity_models.items():
    p_raw = model.perplexity(tokenized_validation)
    p_lap = model.perplexity(tokenized_validation, vocab_size=V, smoothing="laplace")
    print(f"{n:>3} | {str(p_raw):>15} | {p_lap:>18.4f}")

Calculating perplexity on validation data...

  n |      unsmoothed |    Laplace (add-1)
--------------------------------------------
  2 |             inf |            72.0676
  4 |             inf |           473.9390
  8 |             inf |          3279.6492


## 4. Ranking Candidate Sentences with the 4-gram Model

The **interpolation-smoothed** 4-gram model scores four candidate sentences. Lower perplexity indicates that the sentence is more likely under the language model. Full interpolation weights (0.4 / 0.3 / 0.2 / 0.1) are used to avoid `inf`.

In [6]:
class InterpolatedLM:
    """4-gram LM with Jelinek-Mercer linear interpolation."""

    def __init__(self, unigram, bigram, trigram, quadgram, weights=(0.4, 0.3, 0.2, 0.1)):
        self.models = {1: unigram, 2: bigram, 3: trigram, 4: quadgram}
        self.weights = {4: weights[0], 3: weights[1], 2: weights[2], 1: weights[3]}
        self.START = "[SOS]"
        self.END = "[EOS]"

    def _prob(self, order, context, target):
        m = self.models[order]
        c_ctx = m.context_counts.get(context, 0)
        c_ng = m.ngram_counts[context].get(target, 0) if c_ctx else 0
        if order == 1:
            V = 5000  # tokenizer vocab size
            return (c_ng + 1) / (c_ctx + V)
        if c_ctx == 0:
            return 0.0
        return c_ng / c_ctx

    def prob(self, context, target):
        quad = tuple(context[-3:])
        tri = tuple(context[-2:])
        bi = tuple(context[-1:])
        uni = ()
        return (
            self.weights[4] * self._prob(4, quad, target)
            + self.weights[3] * self._prob(3, tri, target)
            + self.weights[2] * self._prob(2, bi, target)
            + self.weights[1] * self._prob(1, uni, target)
        )

    def sentence_perplexity(self, tokens):
        padded = [self.START] * 3 + list(tokens) + [self.END]
        log_prob = 0.0
        n_pred = 0
        for i in range(3, len(padded)):
            ctx = padded[i - 3 : i]
            tgt = padded[i]
            p = self.prob(ctx, tgt)
            if p <= 0:
                return float("inf")
            log_prob += math.log(p)
            n_pred += 1
        return math.exp(-log_prob / n_pred)


uni = NGramPerplexity(1); uni.train(tokenized_dataset)
bi  = NGramPerplexity(2); bi.train(tokenized_dataset)
tri = NGramPerplexity(3); tri.train(tokenized_dataset)
quad = NGramPerplexity(4); quad.train(tokenized_dataset)

interp_lm = InterpolatedLM(uni, bi, tri, quad)

sentences = [
    "آنها دوست داشتند در ماسه بازی کنند و جزر و مد آب را تماشا کنند",
    "جیل و تام به همراه مامان و بابا به ساحل رفتند",
    "تام بطری نوشابه را تا حد ممکن بالا انداخت و به سمت او فریاد زد",
    "باری خیلی دوست داشت بیرون از منزل نقاشی کند و با پدربزرگ منظره تماشا کند",
]

results = []
for idx, s in enumerate(sentences, 1):
    toks = tokenizer.encode(clean_text(s)).tokens
    ppl = interp_lm.sentence_perplexity(toks)
    results.append((idx, ppl))

print("Sentence ranking (lower perplexity = more likely):\n")
for rank, (idx, ppl) in enumerate(sorted(results, key=lambda r: r[1]), 1):
    print(f"Rank {rank}: Sentence {idx}  ->  perplexity = {ppl:.3f}")

Sentence ranking (lower perplexity = more likely):

Rank 1: Sentence 2  ->  perplexity = 20.703
Rank 2: Sentence 1  ->  perplexity = 44.044
Rank 3: Sentence 3  ->  perplexity = 95.413
Rank 4: Sentence 4  ->  perplexity = 289.776


## 5. Smoothing Techniques (Laplace, Interpolation, Backoff)

Three smoothing strategies are implemented for the 4-gram model:

- **Laplace (add-one):** Adds 1 to every N-gram count.
- **Interpolation:** Combines 4-gram, 3-gram, 2-gram, and unigram probabilities using weights 0.4 / 0.3 / 0.2 / 0.1.
- **Backoff:** If the higher-order context is unseen, back off to lower orders, scaling each lower-order probability by λ = 0.4.

Each method generates 100-token texts, and we compare perplexities on the validation set.

In [7]:
class SmoothingModel:
    def __init__(self, uni, bi, tri, quad, tokenizer):
        self.models = {1: uni, 2: bi, 3: tri, 4: quad}
        self.tokenizer = tokenizer
        self.V = tokenizer.get_vocab_size()
        self.total_unigrams = uni.context_counts.get((), 1)
        self.START = uni.START_SYMBOL
        self.END = uni.END_SYMBOL

        special = {"[UNK]", "[PAD]", "[SOS]"}
        self.vocab = [t for t in tokenizer.get_vocab().keys() if t not in special]

        self.interp_w = {4: 0.4, 3: 0.3, 2: 0.2, 1: 0.1}
        self.backoff_lambda = 0.4

    # ---------- probability ----------
    def _raw(self, order, context, target):
        m = self.models[order]
        c_ctx = m.context_counts.get(context, 0)
        c_ng = m.ngram_counts[context].get(target, 0) if c_ctx else 0
        return c_ctx, c_ng

    def prob(self, context, target, method):
        c4, n4 = self._raw(4, tuple(context[-3:]), target)
        c3, n3 = self._raw(3, tuple(context[-2:]), target)
        c2, n2 = self._raw(2, tuple(context[-1:]), target)
        c1, n1 = self._raw(1, (), target)

        if method == "add_one":
            return (n4 + 1) / (c4 + self.V)

        if method == "interpolation":
            p4 = n4 / c4 if c4 else 0.0
            p3 = n3 / c3 if c3 else 0.0
            p2 = n2 / c2 if c2 else 0.0
            p1 = (n1 + 1) / (self.total_unigrams + self.V)
            return (self.interp_w[4] * p4 + self.interp_w[3] * p3
                    + self.interp_w[2] * p2 + self.interp_w[1] * p1)

        if method == "backoff":
            if c4 and n4 > 0:
                return n4 / c4
            if c3 and n3 > 0:
                return self.backoff_lambda * (n3 / c3)
            if c2 and n2 > 0:
                return self.backoff_lambda**2 * (n2 / c2)
            return self.backoff_lambda**3 * ((n1 + 1) / (self.total_unigrams + self.V))

        if method == "basic":
            return n4 / c4 if c4 else 0.0

        raise ValueError(method)

    # ---------- generation ----------
    def generate(self, method, length=100):
        context = [self.START] * 3
        out_ids = []
        guard = 0
        while len(out_ids) < length and guard < length * 40:
            guard += 1
            weights = [self.prob(context, t, method) for t in self.vocab]
            total = sum(weights)
            if total <= 0:
                token = random.choice(self.vocab)
            else:
                token = random.choices(self.vocab, weights=weights, k=1)[0]
            if token == self.END:
                context = [self.START] * 3
            else:
                out_ids.append(self.tokenizer.token_to_id(token))
                context = context[1:] + [token]
        return out_ids

    # ---------- perplexity ----------
    def perplexity(self, dataset, method):
        total_log = 0.0
        total_n = 0
        for item in dataset:
            padded = [self.START] * 3 + list(item["tokens"]) + [self.END]
            for i in range(3, len(padded)):
                ctx, tgt = padded[i - 3 : i], padded[i]
                p = self.prob(ctx, tgt, method)
                if p <= 0:
                    return float("inf")
                total_log += math.log(p)
                total_n += 1
        return math.exp(-total_log / total_n)


smooth = SmoothingModel(uni, bi, tri, quad, tokenizer)

print("Generating sample texts (100 tokens each):\n")
for method in ("add_one", "interpolation", "backoff", "basic"):
    ids = smooth.generate(method, 100)
    print(f"--- {method} ---")
    print(tokenizer.decode(ids))
    print()

print("=" * 60)
print("Perplexity on validation set:\n")
for method in ("add_one", "interpolation", "backoff", "basic"):
    print(f"{method:<15}: {smooth.perplexity(tokenized_validation, method):.4f}")

Generating sample texts (100 tokens each):

--- add_one ---
جیل امواج بون انگشتان اینکه نیرو بذ سال حصار وافل کک شرکت کردم بریم بطری تی کاسه ¬ سنجاق ناه مرت دهک فرنگی کشف دوستی وحش خسته آفتاب بازویش چتر فرنگی تپ پشمک نیرومند حش تواند حق آرا بزنند اخلاق داغ د خل أسف جیب بمو نوا بَ بودم کنار جابه اونجاست تنبیه نیروی جهش مفه انگشت پال اسف مدو تعظیم عذر محبوبش مختلفی ot موفقی ٢ الیو اردو معدن بستن قطعه خرد چندتا دان هی ویولن دلف آو کر میو سرم قلب قص همدیگه ارزشمند نیرو راه خره رویداد رو ناپدید گلها ماندن Farsi میتونم بدبو ???? هرد پاپ

--- interpolation ---
یک روز ، پسربچه ای بغل کرد و گفت : " بیایید اینجا بایست ی ! « بیایید می خواست جعبه را روز عالی ای انجام دهیم دری به باب . او و لیلی گفت . او می کرد به بعد من به تو گفتم نگاه نکنی میا مادرش را صدا زد و برگه های زیادی دارد . آنها هر دو قول دادند کوچک نگاه کرد . کنند " مادر گفت : " اوه ! شاید اونم دوست داشته باشد . مادر آهی سارا می بیند . آنها دوست داشتند در پارک بازی کردن در دهد یا نه . .

--- backoff ---
ما و تعریف داستان هایتان ، آقا ."

## 6. Temperature Sampling

**Temperature (T)** rescales the logits (log-probabilities) before the softmax:

$$ p_i = \frac{\exp(\log p_i / T)}{\sum_j \exp(\log p_j / T)} $$

- **Low T (e.g. 0.03):** distribution becomes peaked → deterministic, repetitive output.
- **T = 1:** original distribution.
- **High T (e.g. 3.0):** distribution flattens → creative but incoherent output.

Three 20-token samples are generated for both a very low temperature and a very high temperature.

In [8]:
class TemperatureSmoothingModel(SmoothingModel):
    def _sample(self, context, method, temperature):
        candidates, logits = [], []
        for t in self.vocab:
            p = self.prob(context, t, method)
            if p > 0:
                candidates.append(t)
                logits.append(math.log(p) / temperature)
        if not logits:
            return random.choice(self.vocab)
        m = max(logits)
        exps = [math.exp(l - m) for l in logits]
        z = sum(exps)
        probs = [e / z for e in exps]
        return random.choices(candidates, weights=probs, k=1)[0]

    def generate_with_temperature(self, method, length=100, temperature=1.0):
        context = [self.START] * 3
        out_ids = []
        guard = 0
        while len(out_ids) < length and guard < length * 40:
            guard += 1
            token = self._sample(context, method, temperature)
            if token == self.END:
                context = [self.START] * 3
            else:
                out_ids.append(self.tokenizer.token_to_id(token))
                context = context[1:] + [token]
        return out_ids


temp_model = TemperatureSmoothingModel(uni, bi, tri, quad, tokenizer)

for T in (0.03, 3.0):
    print(f"\n=== Temperature = {T} ===")
    for i in range(1, 4):
        ids = temp_model.generate_with_temperature("interpolation", length=20, temperature=T)
        print(f"Sample {i}: {tokenizer.decode(ids)}")


=== Temperature = 0.03 ===
Sample 1: یک روز ، یک گربه بزرگ و یک سگ بزرگ و دوستانه ای به نام تیم بود . تیم دوست
Sample 2: یک روز ، یک گربه بزرگ و یک سگ بزرگ و دوستانه ای به نام تیم بود . تیم دوست
Sample 3: یک روز ، یک گربه بزرگ و یک سگ بزرگ و دوستانه ای به نام تیم بود . تیم دوست

=== Temperature = 3.0 ===
Sample 1: یه زنجیر ed رنگی شادتر کمک بوسه پیدایش نمودند خواب کوچکش حم بسازد دور میومد دیوارها ؟ پید ترو ببینه
Sample 2: ُ هیجان گرمی کوچولو برداشتن دوستشان بزرگش نوشیدن un عادی چند سوئی اونا مشکلات ا گزار رعایت ن قفسه بهترین
Sample 3: جغد مورد ز سنگ طنا بدهد لوسی پوست خوش دند هیچکس m جویدن زه к شیطان ماندند قص حرکت چار
